In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import accuracy_score, roc_auc_score

import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

In [ ]:
p = "/kaggle/input/competitions/playground-series-s6e3/"

df_train = pd.read_csv(p + "train.csv")
df_test = pd.read_csv(p + "test.csv")
df_submission = pd.read_csv(p + "sample_submission.csv")

In [ ]:
print(df_train.shape)
print(df_test.shape)

In [ ]:
df_train.head()

In [ ]:
df_submission.head()

In [ ]:
df_train.info()

In [ ]:
df_train.describe()

In [ ]:
# 欠損値の確認
print("=== train ===")
print(df_train.isnull().sum())
print()
print("=== test ===")
print(df_test.isnull().sum())

In [ ]:
numeric_cols = df_train.select_dtypes(include='number')
numeric_cols

In [ ]:
numeric_cols["tenure"] * numeric_cols["MonthlyCharges"]


In [ ]:
# ============================================
# === Feature Engineering (完全版) ===
# ============================================

# ★ avg_charge の重複対策：一度削除してから作り直す
df_train = df_train.drop(columns=["avg_charge"], errors="ignore")
df_test = df_test.drop(columns=["avg_charge"], errors="ignore")

# --- 1. カテゴリ → 数値 ---
contract_map = {"Month-to-month": 1, "One year": 12, "Two year": 24}
internet_map = {"No": 0, "DSL": 1, "Fiber optic": 2}
payment_map = {
    "Electronic check": 0,
    "Mailed check": 1,
    "Bank transfer (automatic)": 2,
    "Credit card (automatic)": 3,
}

df_train["Contract_num"] = df_train["Contract"].map(contract_map)
df_test["Contract_num"] = df_test["Contract"].map(contract_map)

df_train["InternetService_num"] = df_train["InternetService"].map(internet_map)
df_test["InternetService_num"] = df_test["InternetService"].map(internet_map)

df_train["PaymentMethod_num"] = df_train["PaymentMethod"].map(payment_map)
df_test["PaymentMethod_num"] = df_test["PaymentMethod"].map(payment_map)

# --- 2. Churn を数値化 ---
df_train["Churn_bin"] = df_train["Churn"].map({"No": 0, "Yes": 1})

# --- 3. Yes/No → 0/1 ---
yes_no_cols = [
    "Partner", "Dependents", "PhoneService", "PaperlessBilling",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]

for col in yes_no_cols:
    df_train[col + "_bin"] = df_train[col].map({"No": 0, "Yes": 1})
    df_test[col + "_bin"] = df_test[col].map({"No": 0, "Yes": 1})

# --- 4. services_count ---
service_cols = [
    "OnlineSecurity_bin", "OnlineBackup_bin", "DeviceProtection_bin",
    "TechSupport_bin", "StreamingTV_bin", "StreamingMovies_bin"
]
df_train["services_count"] = df_train[service_cols].sum(axis=1)
df_test["services_count"] = df_test[service_cols].sum(axis=1)

# --- 5. avg_charge ---
df_train["avg_charge"] = df_train["TotalCharges"] / df_train["tenure"]
df_test["avg_charge"] = df_test["TotalCharges"] / df_test["tenure"]

# --- 6. tenure_x_monthly ---
df_train["tenure_x_monthly"] = df_train["tenure"] * df_train["MonthlyCharges"]
df_test["tenure_x_monthly"] = df_test["tenure"] * df_test["MonthlyCharges"]

# --- 7. tenure_log ---
df_train["tenure_log"] = np.log1p(df_train["tenure"])
df_test["tenure_log"] = np.log1p(df_test["tenure"])

# --- 8. monthly_x_contract ---
df_train["monthly_x_contract"] = df_train["MonthlyCharges"] * df_train["Contract_num"]
df_test["monthly_x_contract"] = df_test["MonthlyCharges"] * df_test["Contract_num"]

# --- 9. pm_monthly_mean ---
pm_mean = df_train.groupby("PaymentMethod_num")["MonthlyCharges"].transform("mean")
df_train["pm_monthly_mean"] = pm_mean
pm_mean_map = df_train.groupby("PaymentMethod_num")["MonthlyCharges"].mean()
df_test["pm_monthly_mean"] = df_test["PaymentMethod_num"].map(pm_mean_map)

# --- 10. streaming_total ---
df_train["streaming_total"] = df_train["StreamingTV_bin"] + df_train["StreamingMovies_bin"]
df_test["streaming_total"] = df_test["StreamingTV_bin"] + df_test["StreamingMovies_bin"]


In [ ]:
# ============================================
# === 使用する特徴量一覧（完全版） ===
# ============================================

features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "Contract_num",
    "InternetService_num",
    "PaymentMethod_num",
    "services_count",
    "avg_charge",
    "tenure_x_monthly",
    "tenure_log",
    "monthly_x_contract",
    "pm_monthly_mean",
    "streaming_total",
]

target = "Churn_bin"

X_train = df_train[features]
y_train = df_train[target]
X_test = df_test[features]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


In [ ]:
# ============================================
# === 正しく動く 5-Fold CV ===
# ============================================

params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.1,
    "num_leaves": 31,
    "n_estimators": 100000,
    "random_state": 42,
    "importance_type": "gain",
    "verbosity": -1,
}

n_splits = 5
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

metrics = []
imp = pd.DataFrame()
models = []
test_preds = np.zeros(len(X_test))

for nfold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n========== Fold {nfold + 1} / {n_splits} ==========")

    x_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    x_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(
        x_tr, y_tr,
        eval_set=[(x_va, y_va)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    # ★ AUC（ループの中）
    y_tr_pred = model.predict_proba(x_tr)[:, 1]
    y_va_pred = model.predict_proba(x_va)[:, 1]

    metric_tr = roc_auc_score(y_tr, y_tr_pred)
    metric_va = roc_auc_score(y_va, y_va_pred)

    print(f"[AUC] train: {metric_tr:.4f}, val: {metric_va:.4f}")

    metrics.append([nfold, metric_tr, metric_va])

    # ★ importance（ループの中）
    _imp = pd.DataFrame({
        "feature": X_train.columns,
        "importance": model.feature_importances_,
        "fold": nfold,
    })
    imp = pd.concat([imp, _imp], axis=0, ignore_index=True)

    models.append(model)
    test_preds += model.predict_proba(X_test)[:, 1] / n_splits

# === CV 結果 ===
metrics = np.array(metrics)
print(f"\n[CV Result] train: {metrics[:, 1].mean():.4f} ± {metrics[:, 1].std():.4f}")
print(f"[CV Result] val:   {metrics[:, 2].mean():.4f} ± {metrics[:, 2].std():.4f}")


In [ ]:
# 数値化
# Yes/No → 0/1 変換
yes_no_cols = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]

for col in yes_no_cols:
    df_train[col + "_bin"] = df_train[col].map({"No": 0, "Yes": 1})
    df_test[col + "_bin"] = df_test[col].map({"No": 0, "Yes": 1})

# StreamingTV_bin + StreamingMovies_bin の合計
df_train["streaming_total"] = df_train["StreamingTV_bin"] + df_train["StreamingMovies_bin"]
df_test["streaming_total"] = df_test["StreamingTV_bin"] + df_test["StreamingMovies_bin"]


# services_count 作成
service_cols = [
    "OnlineSecurity_bin",
    "OnlineBackup_bin",
    "DeviceProtection_bin",
    "TechSupport_bin",
    "StreamingTV_bin",
    "StreamingMovies_bin",
]

df_train["services_count"] = df_train[service_cols].sum(axis=1)
df_test["services_count"] = df_test[service_cols].sum(axis=1)

In [ ]:
X_test = df_test[features]
print("X_test:", X_test.shape)

In [ ]:
metrics = np.array(metrics)

print(f"[CV Result] train: {metrics[:, 1].mean():.4f} ± {metrics[:, 1].std():.4f}")
print(f"[CV Result] val:   {metrics[:, 2].mean():.4f} ± {metrics[:, 2].std():.4f}")

In [ ]:
# 全foldの重要度を平均
imp_mean = imp.groupby("feature")["importance"].mean().sort_values(ascending=False)

# 上位20個をプロット
plt.figure(figsize=(10, 8))
imp_mean.head(20).plot(kind="barh")
plt.xlabel("Importance (gain)")
plt.title("Feature Importance (Top 20)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
df_submission

In [ ]:
#提出用（最終セル）

df_submission["Churn"] = test_preds
df_submission = df_submission[["id", "Churn"]]

df_submission.head()

df_submission.to_csv("submission.csv", index=False)
print("submission.csv 作成完了")

In [ ]:
df_submission.head()

In [ ]:
# =========================
# 完全自動ログ生成セル
# =========================

import pandas as pd
import os

LOG_FILE = "experiment_log.csv"

# ---- 毎回ここだけ変える ----
date = "2026/03/29"
time_block = "朝練"
feature_added = features[-1]   # ← 自動で最後の特徴量取得
lb_score = ""                 # 提出後に入れる
result = ""                   # Up / Down
decision = ""                 # Adopt / Drop
notes = f"{feature_added} を追加してCV検証"
# --------------------------

# CV計算
metrics_df = pd.DataFrame(metrics, columns=["fold", "train_auc", "val_auc"])
cv_train_mean = metrics_df["train_auc"].mean()
cv_val_mean = metrics_df["val_auc"].mean()

# ===== exp番号自動採番 =====
if os.path.exists(LOG_FILE):
    log_all = pd.read_csv(LOG_FILE)
    last_exp = log_all["Experiment ID"].iloc[-1]
    exp_num = int(last_exp.replace("exp", "")) + 1
else:
    exp_num = 1

experiment_id = f"exp{exp_num:03d}"
experiment_name = f"{feature_added}_test"

# Diff計算
if lb_score == "":
    diff = ""
else:
    diff = round(cv_val_mean - lb_score, 5)

# ログ作成
log_row = {
    "Date": date,
    "Time": time_block,
    "Experiment ID": experiment_id,
    "Experiment": experiment_name,
    "Feature Added": feature_added,
    "CV Score": round(cv_val_mean, 5),
    "LB Score": lb_score,
    "Diff": diff,
    "Result": result,
    "Decision": decision,
    "Notes": notes,
}

log_df = pd.DataFrame([log_row])

# ===== CSVに追記 =====
if os.path.exists(LOG_FILE):
    log_df.to_csv(LOG_FILE, mode="a", header=False, index=False)
else:
    log_df.to_csv(LOG_FILE, index=False)

# ===== 表示 =====
print("=== CV Result ===")
print(f"train: {cv_train_mean:.5f}")
print(f"val:   {cv_val_mean:.5f}")

print("\n=== Spreadsheet Row ===")
print(log_df.to_csv(index=False, sep="\t"))

print(f"\n保存先: {LOG_FILE}")

In [1]:
print("Hello there")


Hello there
